<a href="https://colab.research.google.com/github/AshokGit544/Azure-Data-Lake-Storage-Lakehouse-Zoning/blob/main/Azure_Data_Lake_Storage_Lakehouse_Zoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import json

import pandas as pd
import numpy as np

BASE_PATH = Path('/content/azure_data_lake_storage_lakehouse_zoning_demo')
RAW_PATH = BASE_PATH / 'data' / 'raw'
STANDARDIZED_PATH = BASE_PATH / 'data' / 'standardized'
CURATED_PATH = BASE_PATH / 'data' / 'curated'
OUTPUT_PATH = BASE_PATH / 'outputs'
ARCH_PATH = BASE_PATH / 'architecture'

for p in [RAW_PATH, STANDARDIZED_PATH, CURATED_PATH, OUTPUT_PATH, ARCH_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/azure_data_lake_storage_lakehouse_zoning_demo


In [2]:
random.seed(42)
num_customers = 1200
num_billing = 3500
num_meter = 14000
num_outages = 1000
regions = ['North', 'South', 'East', 'West', 'Central']
start_date = datetime(2024, 1, 1)

# raw customer data
customers = []
for i in range(1, num_customers + 1):
    customers.append({
        'customer_id': f'CUST{i:05d}',
        'customer_name': f'Customer_{i}',
        'region': random.choice(regions),
        'customer_type': random.choice(['Residential', 'Commercial', 'Industrial']),
        'source_system': 'CustomerSystem'
    })

pd.DataFrame(customers).to_csv(RAW_PATH / 'customers_raw.csv', index=False)

# raw billing / sap fico style data
billing = []
for i in range(1, num_billing + 1):
    cust = random.randint(1, num_customers)
    bill_date = start_date + timedelta(days=random.randint(0, 179))
    billing.append({
        'billing_id': f'BILL{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'bill_date': bill_date.strftime('%Y-%m-%d'),
        'bill_amount': round(float(max(40, np.random.normal(145, 40))), 2),
        'sap_fico_doc_id': f'SAPFICO{i:08d}',
        'source_system': 'SAPFICO'
    })

billing_df = pd.DataFrame(billing)
billing_df.loc[10, 'bill_amount'] = None
billing_df.to_csv(RAW_PATH / 'billing_raw.csv', index=False)

# raw smart meter data
meter = []
for i in range(1, num_meter + 1):
    cust = random.randint(1, num_customers)
    read_date = start_date + timedelta(days=random.randint(0, 179))
    meter.append({
        'meter_record_id': f'MTRREC{i:07d}',
        'customer_id': f'CUST{cust:05d}',
        'reading_date': read_date.strftime('%Y-%m-%d'),
        'kwh_usage': round(float(max(5, np.random.normal(35, 10))), 2),
        'estimated_read_flag': random.choice([0, 0, 0, 1]),
        'source_system': 'SmartMeterSystem'
    })

meter_df = pd.DataFrame(meter)
meter_df = pd.concat([meter_df, meter_df.iloc[:25]], ignore_index=True)
meter_df.to_csv(RAW_PATH / 'meter_usage_raw.csv', index=False)

# raw outage data
outages = []
for i in range(1, num_outages + 1):
    cust = random.randint(1, num_customers)
    outage_date = start_date + timedelta(days=random.randint(0, 179))
    outages.append({
        'outage_id': f'OUT{i:06d}',
        'customer_id': f'CUST{cust:05d}',
        'outage_date': outage_date.strftime('%Y-%m-%d'),
        'duration_minutes': max(5, int(np.random.normal(80, 25))),
        'source_system': 'OutageSystem'
    })

pd.DataFrame(outages).to_csv(RAW_PATH / 'outages_raw.csv', index=False)

print('Raw zone files created successfully.')
for file in RAW_PATH.glob('*.csv'):
    print(file.name)

Raw zone files created successfully.
meter_usage_raw.csv
billing_raw.csv
customers_raw.csv
outages_raw.csv


In [3]:
storage_design = {
    'raw_zone': 'Stores source data exactly as received from enterprise systems',
    'standardized_zone': 'Stores cleaned, standardized, and reusable datasets',
    'curated_zone': 'Stores business-ready integrated datasets for reporting, analytics, ML, and AI-ready consumption',
    'zones': [
        {'zone': 'raw', 'purpose': 'source traceability and auditability'},
        {'zone': 'standardized', 'purpose': 'cleaned and reusable datasets'},
        {'zone': 'curated', 'purpose': 'integrated business-ready outputs'}
    ]
}

with open(ARCH_PATH / 'storage_zone_design.json', 'w') as f:
    json.dump(storage_design, f, indent=2)

print(json.dumps(storage_design, indent=2))

{
  "raw_zone": "Stores source data exactly as received from enterprise systems",
  "standardized_zone": "Stores cleaned, standardized, and reusable datasets",
  "curated_zone": "Stores business-ready integrated datasets for reporting, analytics, ML, and AI-ready consumption",
  "zones": [
    {
      "zone": "raw",
      "purpose": "source traceability and auditability"
    },
    {
      "zone": "standardized",
      "purpose": "cleaned and reusable datasets"
    },
    {
      "zone": "curated",
      "purpose": "integrated business-ready outputs"
    }
  ]
}


In [4]:
lineage_logs = []

def log_lineage(source_file, target_file, source_zone, target_zone, row_count, message=''):
    lineage_logs.append({
        'source_file': source_file,
        'target_file': target_file,
        'source_zone': source_zone,
        'target_zone': target_zone,
        'row_count': row_count,
        'timestamp': datetime.now().isoformat(),
        'message': message
    })

# customers
customers_std = pd.read_csv(RAW_PATH / 'customers_raw.csv').drop_duplicates(subset=['customer_id'])
customers_std.to_csv(STANDARDIZED_PATH / 'customers_standardized.csv', index=False)
log_lineage('customers_raw.csv', 'customers_standardized.csv', 'raw', 'standardized', len(customers_std), 'Deduplicated customer data')

# billing
billing_std = pd.read_csv(RAW_PATH / 'billing_raw.csv')
billing_std['bill_amount'] = billing_std['bill_amount'].fillna(0)
billing_std = billing_std[billing_std['bill_amount'] >= 0]
billing_std.to_csv(STANDARDIZED_PATH / 'billing_standardized.csv', index=False)
log_lineage('billing_raw.csv', 'billing_standardized.csv', 'raw', 'standardized', len(billing_std), 'Handled nulls and filtered invalid bill amounts')

# meter
meter_std = pd.read_csv(RAW_PATH / 'meter_usage_raw.csv')
meter_std = meter_std.drop_duplicates(subset=['customer_id', 'reading_date'])
meter_std.to_csv(STANDARDIZED_PATH / 'meter_usage_standardized.csv', index=False)
log_lineage('meter_usage_raw.csv', 'meter_usage_standardized.csv', 'raw', 'standardized', len(meter_std), 'Removed duplicate meter records')

# outages
outages_std = pd.read_csv(RAW_PATH / 'outages_raw.csv')
outages_std = outages_std[outages_std['duration_minutes'] >= 0]
outages_std.to_csv(STANDARDIZED_PATH / 'outages_standardized.csv', index=False)
log_lineage('outages_raw.csv', 'outages_standardized.csv', 'raw', 'standardized', len(outages_std), 'Validated outage durations')

print('Standardized zone files created successfully.')
for file in STANDARDIZED_PATH.glob('*.csv'):
    print(file.name)

Standardized zone files created successfully.
customers_standardized.csv
billing_standardized.csv
outages_standardized.csv
meter_usage_standardized.csv


In [5]:
customers_cur = pd.read_csv(STANDARDIZED_PATH / 'customers_standardized.csv')
billing_cur = pd.read_csv(STANDARDIZED_PATH / 'billing_standardized.csv')
meter_cur = pd.read_csv(STANDARDIZED_PATH / 'meter_usage_standardized.csv')
outages_cur = pd.read_csv(STANDARDIZED_PATH / 'outages_standardized.csv')

meter_daily = meter_cur.groupby(['customer_id', 'reading_date'], as_index=False).agg(
    daily_kwh_usage=('kwh_usage', 'sum')
).rename(columns={'reading_date': 'event_date'})

billing_daily = billing_cur.groupby(['customer_id', 'bill_date'], as_index=False).agg(
    daily_billed_amount=('bill_amount', 'sum'),
    daily_bill_count=('billing_id', 'count')
).rename(columns={'bill_date': 'event_date'})

outage_daily = outages_cur.groupby(['customer_id', 'outage_date'], as_index=False).agg(
    daily_outage_count=('outage_id', 'count'),
    daily_outage_minutes=('duration_minutes', 'sum')
).rename(columns={'outage_date': 'event_date'})

curated_df = meter_daily.merge(customers_cur, on='customer_id', how='left')
curated_df = curated_df.merge(billing_daily, on=['customer_id', 'event_date'], how='left')
curated_df = curated_df.merge(outage_daily, on=['customer_id', 'event_date'], how='left')

curated_df['daily_billed_amount'] = curated_df['daily_billed_amount'].fillna(0)
curated_df['daily_bill_count'] = curated_df['daily_bill_count'].fillna(0)
curated_df['daily_outage_count'] = curated_df['daily_outage_count'].fillna(0)
curated_df['daily_outage_minutes'] = curated_df['daily_outage_minutes'].fillna(0)

curated_df.to_csv(CURATED_PATH / 'enterprise_curated_dataset.csv', index=False)
log_lineage('standardized zone datasets', 'enterprise_curated_dataset.csv', 'standardized', 'curated', len(curated_df), 'Built integrated business-ready curated dataset')

print('Curated zone file created successfully.')
print(CURATED_PATH / 'enterprise_curated_dataset.csv')

Curated zone file created successfully.
/content/azure_data_lake_storage_lakehouse_zoning_demo/data/curated/enterprise_curated_dataset.csv


In [6]:
kpi_df = curated_df.groupby('region', as_index=False).agg(
    avg_daily_kwh_usage=('daily_kwh_usage', 'mean'),
    avg_daily_outage_minutes=('daily_outage_minutes', 'mean'),
    total_billed_amount=('daily_billed_amount', 'sum')
)

kpi_df.to_csv(OUTPUT_PATH / 'curated_kpis.csv', index=False)
kpi_df.head()

,region,avg_daily_kwh_usage,avg_daily_outage_minutes,total_billed_amount
0,Central,35.094463,0.384697,5388.87
1,East,34.943103,0.465201,6426.93
2,North,35.092924,0.203540,6923.37
3,South,35.022760,0.365028,5040.40
4,West,35.070051,0.454514,6468.70


In [7]:
lineage_log_df = pd.DataFrame(lineage_logs)
lineage_log_df.to_csv(OUTPUT_PATH / 'lineage_log.csv', index=False)
lineage_log_df

,source_file,target_file,source_zone,target_zone,row_count,timestamp,message
0,customers_raw.csv,customers_standardized.csv,raw,standardized,1200,2026-03-17T04:46:19.135494,Deduplicated customer data
1,billing_raw.csv,billing_standardized.csv,raw,standardized,3500,2026-03-17T04:46:19.201533,Handled nulls and filtered invalid bill amounts
2,meter_usage_raw.csv,meter_usage_standardized.csv,raw,standardized,13547,2026-03-17T04:46:19.386718,Removed duplicate meter records
3,outages_raw.csv,outages_standardized.csv,raw,standardized,1000,2026-03-17T04:46:19.418366,Validated outage durations
4,standardized zone datasets,enterprise_curated_dataset.csv,standardized,curated,13547,2026-03-17T04:47:43.326432,Built integrated business-ready curated dataset


In [8]:
summary = {
    'generated_at': datetime.now().isoformat(),
    'raw_files': len(list(RAW_PATH.glob('*.csv'))),
    'standardized_files': len(list(STANDARDIZED_PATH.glob('*.csv'))),
    'curated_files': len(list(CURATED_PATH.glob('*.csv'))),
    'raw_zone_records': sum(len(pd.read_csv(f)) for f in RAW_PATH.glob('*.csv')),
    'standardized_zone_records': sum(len(pd.read_csv(f)) for f in STANDARDIZED_PATH.glob('*.csv')),
    'curated_zone_records': len(pd.read_csv(CURATED_PATH / 'enterprise_curated_dataset.csv'))
}

with open(OUTPUT_PATH / 'storage_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "generated_at": "2026-03-17T04:49:08.839294",
  "raw_files": 4,
  "standardized_files": 4,
  "curated_files": 1,
  "raw_zone_records": 19725,
  "standardized_zone_records": 19247,
  "curated_zone_records": 13547
}


In [9]:
pd.read_csv(CURATED_PATH / 'enterprise_curated_dataset.csv').head()

,customer_id,event_date,daily_kwh_usage,customer_name,region,customer_type,source_system,daily_billed_amount,daily_bill_count,daily_outage_count,daily_outage_minutes
0,CUST00001,2024-01-01,36.27,Customer_1,North,Residential,CustomerSystem,0.0,0.0,0.0,0.0
1,CUST00001,2024-01-08,34.50,Customer_1,North,Residential,CustomerSystem,0.0,0.0,0.0,0.0
2,CUST00001,2024-03-03,28.40,Customer_1,North,Residential,CustomerSystem,0.0,0.0,0.0,0.0
3,CUST00001,2024-03-04,21.70,Customer_1,North,Residential,CustomerSystem,0.0,0.0,0.0,0.0
4,CUST00001,2024-03-23,29.55,Customer_1,North,Residential,CustomerSystem,0.0,0.0,0.0,0.0
